# പാഠം 01 - എ.ഐ. ഏജന്റുമാർക്ക് പരിചയം

**AI ഏജന്റുമാർക്ക് തുടക്കം** കോഴ്‌സിലെ ആദ്യ പാഠത്തിലേക്ക് സ്വാഗതം!

**AI ഏജന്റ്** എന്നത് ഒരു പ്രോഗ്രാമാണ്, ഇത് വലിയ ഭാഷാ മാതൃക (LLM) എന്നതിനെ തന്റെ തർക്ക ഘടകമായി ഉപയോഗിച്ച് യാഥാർത്ഥ്യ ലോകത്തിൽ *പ്രവർത്തനങ്ങൾ* എടുക്കാൻ കഴിയും — API കളിൽ വിളിക്കൽ, ഡാറ്റാബേസുകൾ അന്വേഷിക്കൽ, അല്ലെങ്കിൽ കോഡ് പ്രവർത്തിപ്പിക്കൽ — ഉപയോക്താവിന്റെ അംഗീകാരത്തിന് ഒരു ലക്ഷ്യം പൂർത്തിയാക്കുന്നതിന്.

ഈ നോട്ടുബുക്കിൽ നിങ്ങൾ നിങ്ങളുടെ ആദ്യ ഏജന്റ് നിർമ്മിക്കും: അവധി ലക്ഷ്യസ്ഥലങ്ങൾ ശുപാർശ ചെയ്യുന്ന **ട്രാവൽ ഏജന്റ്**. യാത്രയിൽ നിങ്ങൾ പഠിക്കാനിരിക്കുന്ന കാര്യങ്ങൾ:

1. **Microsoft Agent Framework** ഉപയോഗിച്ച് Azure AI Foundry Agent Service-ലേക്ക് ബന്ധിപ്പിക്കുക.
2. ഏജന്റിനെ ഒരു **ടൂൾ** നൽകുക — ഏജന്റ് വിളിക്കാവുന്ന ലളിതമായ പൈറ്റൺ ഫംക്ഷൻ.
3. ഏജന്റ് പ്രവർത്തിപ്പിച്ച് അതിന്റെ ഉത്തരം പരിശോധിക്കുക.
4. ഏജന്റിന്റെ ഉത്തരത്തെ ටോക്കൺ-ബൈ-ടോക്കണായി സ്റ്റ്രീം ചെയ്യുക.


## ക്രമീകരണം

ഈ നോട്ട്ബുക്ക് പ്രവർത്തിപ്പിക്കുമ്ബോൾ, നിങ്ങൾക്കുണ്ടെന്ന് ഉറപ്പാക്കുക:

1. **ചര്ച്ച്ച മോഡൽ (ഉദാ: `gpt-4o-mini`) വിന്യസിച്ചിട്ടുള്ള ഒരു Azure AI Foundry പ്രോജക്ട്**.
2. **Azure CLI ഉപയോഗിച്ച് ലോഗിൻ ചെയ്തിട്ടുണ്ട്** — നിങ്ങളുടെ ടെർമിനലിൽ `az login` പ്രവർത്തിപ്പിക്കുക.
3. **ആവശ്യമായ എൻവയോൺമെന്റ് വേരിയബിൾസ് സെറ്റ് ചെയ്തിരിക്കുന്നു:**
   - `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Azure AI Foundry പ്രോജക്ട് എന്റ്പോയിന്റ്.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — വിന്യസിച്ച മോഡലിന്റെ പേരായി.

താഴെ നൽകിയ സെൽ നിങ്ങള്‍ക്ക് ആവശ്യമായ പൈത്തൺ പാക്കേജുകൾ ഇൻസ്റ്റാൾ ചെയ്യും.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## നിങ്ങളുടെ ആദ്യ ഏജന്റ് സൃഷ്ടിക്കുക

ഏജന്റിന് രണ്ട് കാര്യങ്ങൾ ആവശ്യമാണ്:

- അതിന് *ആയാളാണ്* എന്നും *എങ്ങനെ പെരുമരുത്താമ* എന്നും പറഞ്ഞുകൊടുക്കുന്ന **സൂചനകൾ** (ഒരു സിസ്റ്റം പ്രാമ്പ്റ്റ്).
- ഏജന്റ് വിവരങ്ങൾ നേടുകയോ പ്രവർത്തനങ്ങൾ നടത്തുകയോ ചെയ്യാൻ വിളിക്കാവുന്ന `@tool` ആയി അലങ്കരിച്ചിരിക്കുന്ന പൈതൺ ഫംഗ്ഷനുകളായ **ഉപകരണങ്ങൾ**.

താഴെ, പ്രശസ്തമായ വിനോദസഞ്ചാര ഗമ്യസ്ഥലങ്ങളുടെ പട്ടിക നൽകുന്ന ഒരു ലളിതമായ ഉപകരണം നിവേദിക്കുന്നു. ഉപയോക്താവ് യാത്രാ ശുപാർശകൾ ചോദിക്കുന്നപ്പോൾ ഏജന്റ് ഈ ഉപകരണം ഉപയോഗിക്കും.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## സ്റ്റ്രീമിംഗ് പ്രതികരണങ്ങൾ

ഏകദേശം പ്രതികരണമുറപ്പിക്കാൻ ඔබക്ക് ഏജന്റിന്റെ പ്രതികരണം **സ്റ്റ്രീം** ചെയ്യാം. പൂർണ്ണമായ മറുപടി കാത്തിരിക്കുന്നതിനുപകരം, ഏജന്റ് സൃഷ്ടിക്കുന്നതുപോലെ ടെക്‌സ്‌റ്റ് ഭാഗങ്ങൾ നൽകുന്നു. ഇത് പ്രത്യേകിച്ച് ചാറ്റ് ഇന്റർഫേസുകളിൽ പ്രയോജനപ്പെടും, നിങ്ങൾക്ക് മാധ്യമപ്രദേശങ്ങളിൽ വരുമ്പോൾ ഔട്ട്‌പുട്ട് റിയൽ ടൈം കാണിക്കാൻ.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## സംഗ്രഹം

ഈ പാഠത്തില് നിങ്ങൾ പഠിച്ചത്:

- **പ്രൊവൈഡറെ സൃഷ്ടിക്കുക** `AzureAIProjectAgentProvider` വഴി Azure AI Foundry Agent Service-യുമായി ബന്ധിപ്പിക്കുന്നത്.
- **ടൂൾ നിർവചിക്കുക** `@tool` ഡികോറേറ്റർ ഉപയോഗിച്ച്, അതിലൂടെ ഏജന്റ് നിങ്ങളുടെ Python ഫങ്ഷനുകൾ വിളിക്കാൻ കഴിയും.
- **ഏജന്റ് ഓടിക്കുക** ഉപയോഗിക്കുന്ന ഉപയോക്തൃ സന്ദേശത്തോടെ, അതിന്റെ പ്രതികരണം പ്രിന്റുചെയ്യുക.
- **പ്രതികരണങ്ങൾ സ്റ്റ്രീം ചെയ്യുക** യഥാർത്ഥ സമയ ഔട്ട്പുട്ടിനായി.

അടുത്ത പാഠത്തിൽ, ഏജന്റിക് ഫ്രെയിംവർക്കുകളെ കൂടുതൽ ആഴത്തിൽ അന്വേഷിച്ച്, ഏജന്റുകൾക്ക് കൂടുതൽ ശക്തമായ ഉപകരണങ്ങളും ബഹുവിഭാഗ ചിന്തന ശേഷിയും നൽകുന്നത് പഠിക്കും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**പരിഹാര്യം**:  
ഈ രേഖ AI പരിഭാഷ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷ ചെയ്തതാണ്. നാം കാര്യക്ഷമത ലക്ഷ്യമിടുമ്പോഴും, സ്വയം പ്രവർത്തിക്കുന്ന പരിഭാഷയിൽ പിശകുകളോ അശുദ്ധികളോ ഉണ്ടാകാനുള്ള സാധ്യതയുണ്ട്. മാതൃഭാഷയിലുള്ള യഥാർത്ഥ രേഖ അവകാശമുള്ള സെറോ ആയി പരിഗണിക്കേണ്ടതാണ്. അത്യന്താപേക്ഷിത വിവരങ്ങൾക്കായി പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ നിർദ്ദേശിക്കപ്പെടുന്നു. ഈ പരിഭാഷയുടെ ഉപയോഗത്തിൽ നിന്നുണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണകൾക്കും തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കും ഞങ്ങൾ ഉത്തരവാദിത്തം ഏറ്റെടുക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
